# Defend the center: ANN from scratch

Standard `doom_defend_the_center`, not the flat-actions variant. Two categorical
factors control turning (3 choices) and attack (2 choices). Both policies use five
logits, allowing six combinations including turning while shooting.

Use a fresh Linux GPU kernel for each notebook and run from top to bottom.
First run defaults to a 100k-step smoke test with 10 evaluation episodes. For a
full matched comparison set SMOKE_TEST=False in both notebooks (8M steps, 300 episodes).
The SNN does not load ANN weights. Fresh mode refuses existing run directories.
Evaluation uses the full Sample Factory actor-critic with trained normalization;
there is no four-action standalone extraction or inherited basic-scenario score table.


## 1. Server and GPU setup

In [ ]:
import os, sys, json, random, hashlib, time, importlib
from pathlib import Path
from importlib.metadata import version
# This variable also reaches newly spawned Sample Factory checkpoint loaders.
# Only load checkpoints produced by this project or another trusted source.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import numpy as np
import torch
import gymnasium as gym
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "defend the center").is_dir() and (p / "conversion_friendly_v1").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from this repository.")
WORK_DIR = ROOT / "defend the center"
os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path: sys.path.insert(0, str(WORK_DIR))
OUTPUT_ROOT = Path(os.environ.get("DEFEND_OUTPUT_ROOT", str(WORK_DIR / "outputs"))).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(1)
assert torch.cuda.is_available(), "Launch this kernel inside a GPU allocation."
DEVICE = "cuda"
CPU_BUDGET = int(os.environ.get("SLURM_CPUS_PER_TASK", len(os.sched_getaffinity(0))))
NUM_WORKERS = max(1, min(8, CPU_BUDGET, int(os.environ.get("DOOM_NUM_WORKERS", max(1, CPU_BUDGET-2)))))
print("Kernel:", sys.executable, "GPU:", torch.cuda.get_device_name(0), "workers:", NUM_WORKERS)
print("GPU test:", (torch.ones(1, device=DEVICE)+1).item())


## 2. Settings

In [ ]:
ENV_NAME = "doom_defend_the_center"
SEED = 0
RUN_MODE = "fresh"          # fresh, resume, evaluate
RUN_TAG = "v1"              # new tag for each independent run
SMOKE_TEST = True           # switch False for full training
TOTAL_STEPS = 100_000 if SMOKE_TEST else 8_000_000
BATCH_SIZE = 1024
LEARNING_RATE = 1e-4
ENVS_PER_WORKER = 4
N_EVAL_EPISODES = 10 if SMOKE_TEST else 300
EVAL_SEEDS = list(range(10000, 10000 + N_EVAL_EPISODES))
assert RUN_MODE in {"fresh", "resume", "evaluate"}
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
EXPERIMENT = f"defend_ann_seed{SEED}_{RUN_TAG}" + ("_smoke" if SMOKE_TEST else "")


## Register the policy and check the environment

In [ ]:
import functools
from sample_factory.algo.utils.context import global_model_factory
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args, load_from_checkpoint
from sample_factory.envs.create_env import create_env
from sample_factory.envs.env_utils import register_env
from sample_factory.model.actor_critic import create_actor_critic
from sample_factory.algo.utils.rl_utils import prepare_and_normalize_obs
from sample_factory.train import run_rl
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec

for spec in DOOM_ENVS:
    register_env(spec.name, functools.partial(make_doom_env_from_spec, spec))
from sf_examples.vizdoom.doom.doom_model import make_vizdoom_encoder
global_model_factory().register_encoder_factory(make_vizdoom_encoder)

def parse_cfg(argv, evaluation=False):
    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
    add_doom_env_args(parser); doom_override_defaults(parser)
    pass
    return parse_full_cfg(parser, argv)

TRAIN_DIR = str(OUTPUT_ROOT / "train_dir")
RUN_DIR = Path(TRAIN_DIR) / EXPERIMENT
if RUN_MODE == "fresh" and RUN_DIR.exists():
    raise FileExistsError(f"Fresh run refuses to overwrite/resume {RUN_DIR}. Choose a new RUN_TAG.")
if RUN_MODE != "fresh" and not (RUN_DIR / "config.json").is_file():
    raise FileNotFoundError(f"No saved configuration in {RUN_DIR}")
env_cfg = parse_cfg([f"--env={ENV_NAME}", "--experiment=defend_probe",
                     f"--train_dir={TRAIN_DIR}", f"--seed={SEED}",
                     "--res_h=72", "--res_w=128", "--env_frameskip=4"], evaluation=True)
env = create_env(ENV_NAME, cfg=env_cfg, env_config=None)
assert isinstance(env.action_space, gym.spaces.Tuple)
assert [s.n for s in env.action_space.spaces] == [3, 2], env.action_space
ACTION_SPACE = env.action_space
raw_space = env.observation_space
OBS_SPACE = raw_space if isinstance(raw_space, gym.spaces.Dict) else gym.spaces.Dict({"obs": raw_space})
assert OBS_SPACE["obs"].shape == (3, 72, 128)
obs, _ = env.reset(seed=SEED)
env.step(ACTION_SPACE.sample())
env.close()
print("Observation:", OBS_SPACE, "Action:", ACTION_SPACE)
print("Turning: idle/left/right; attack: off/on. Five logits, six combinations.")


## Train with Sample Factory

PPO trains the policy and critic from rewards. Fresh runs initialize weights; resume uses this experiment's saved state. Check the resolved config/logs: resumed settings can differ from requested arguments.

In [ ]:
argv = [f"--env={ENV_NAME}", f"--experiment={EXPERIMENT}", f"--train_dir={TRAIN_DIR}",
        f"--seed={SEED}", "--device=gpu", "--use_rnn=False", "--nonlinearity=relu",
        "--actor_critic_share_weights=True", "--res_h=72", "--res_w=128", "--env_frameskip=4",
        f"--num_workers={NUM_WORKERS}", f"--num_envs_per_worker={ENVS_PER_WORKER}",
        f"--batch_size={BATCH_SIZE}", f"--learning_rate={LEARNING_RATE}",
        f"--train_for_env_steps={TOTAL_STEPS}", "--set_workers_cpu_affinity=False"]

if RUN_MODE != "evaluate":
    cfg = parse_cfg(argv)
    if RUN_MODE == "fresh" and RUN_DIR.exists():
        raise FileExistsError("Run directory appeared; use a new tag or deliberately select resume.")
    t0 = time.time()
    status = run_rl(cfg)
    print("Status:", status, "minutes:", (time.time()-t0)/60)
    assert status == 0, "Training failed; inspect logs before evaluation."
else:
    print("Training skipped: evaluation mode")


## Reload the latest checkpoint

In [ ]:
checkpoints = sorted((RUN_DIR / "checkpoint_p0").glob("checkpoint_*.pth"))
if not checkpoints: raise FileNotFoundError(f"No checkpoints in {RUN_DIR}")
CKPT_PATH = checkpoints[-1]  # latest regular checkpoint, not best-reward checkpoint
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
eval_cfg = parse_cfg([f"--env={ENV_NAME}", f"--experiment={EXPERIMENT}", f"--train_dir={TRAIN_DIR}"], evaluation=True)
eval_cfg = load_from_checkpoint(eval_cfg)
assert eval_cfg.env == ENV_NAME and not eval_cfg.use_rnn
ac = create_actor_critic(eval_cfg, OBS_SPACE, ACTION_SPACE)
ac.load_state_dict(ckpt["model"], strict=True)
ac.model_to_device(torch.device(DEVICE)); ac.eval()

def policy_logits(raw_obs):
    raw_dict = raw_obs if isinstance(raw_obs, dict) else {"obs": raw_obs}
    batch = {k: torch.as_tensor(v).unsqueeze(0) for k,v in raw_dict.items()}
    with torch.no_grad():
        normalized = prepare_and_normalize_obs(ac, batch)
        head = ac.forward_head(normalized)
        core, _ = ac.forward_core(head, None)
        return ac.forward_tail(core, values_only=False, sample_actions=False)["action_logits"]

def tuple_action(logits):
    assert logits.shape == (1, 5)
    return (int(logits[0,:3].argmax()), int(logits[0,3:].argmax()))

# Check both factors independently, including turn-and-shoot combinations.
assert tuple_action(torch.tensor([[0.,2.,1.,0.,3.]])) == (1,1)
print("Loaded:", CKPT_PATH.name, "steps:", ckpt["env_steps"])


## ANN architecture inspection

Inspects the latest loaded checkpoint before gameplay evaluation. Shape tracing uses a dummy image; it is not a performance measurement. Module checks cannot detect every functional operation or prove conversion correctness.


In [ ]:
# Inspect the loaded ANN, including its decoder and five-logit action readout.
# This reports conversion concerns; it does not certify conversion equivalence.
import torch.nn as nn
print(ac)
print("Trainable parameters:", sum(p.numel() for p in ac.parameters() if p.requires_grad))
print("Policy readout:", ac.action_parameterization)
encoder_modules = list(ac.encoder.named_modules())
conv_layers = [(name, m) for name,m in encoder_modules if isinstance(m, nn.Conv2d)]
linear_layers = [(name, m) for name,m in encoder_modules if isinstance(m, nn.Linear)]
relu_names = [name for name,m in encoder_modules if isinstance(m, nn.ReLU)]
batchnorm_names = [name for name,m in ac.named_modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]
other_activation_types = (nn.ELU, nn.LeakyReLU, nn.PReLU, nn.GELU, nn.SiLU, nn.Sigmoid, nn.Tanh, nn.Softmax)
other_activations = [(name,type(m).__name__) for name,m in ac.named_modules() if isinstance(m,other_activation_types)]
maxpool_names = [name for name,m in ac.named_modules() if isinstance(m,(nn.MaxPool1d,nn.MaxPool2d,nn.MaxPool3d))]
shapes = {}
def record_shape(name):
    def hook(module, inputs, output):
        shapes[name] = {"input": list(inputs[0].shape) if inputs and torch.is_tensor(inputs[0]) else None,
                        "output": list(output.shape) if torch.is_tensor(output) else None}
    return hook
handles = [m.register_forward_hook(record_shape(name)) for name,m in ac.named_modules()
           if isinstance(m,(nn.Conv2d,nn.Linear))]
try:
    test_logits = policy_logits(np.zeros((3,72,128),dtype=np.uint8))
finally:
    for handle in handles: handle.remove()
assert test_logits.shape == (1,5), test_logits.shape
print("Encoder convolutions:", [(name,str(m)) for name,m in conv_layers])
print("Encoder linear layers:", [(name,str(m)) for name,m in linear_layers])
print("Layer shapes:", json.dumps(shapes,indent=2))
concerns = []
if batchnorm_names: concerns.append("BatchNorm present: fold/evaluate it explicitly before conversion.")
if other_activations: concerns.append("Non-ReLU activations present: not covered by our ReLU-to-IF conversion.")
if maxpool_names: concerns.append("Max pooling present: requires an explicit conversion strategy.")
if eval_cfg.use_rnn: concerns.append("Recurrent core present: our stateless conversion does not cover it.")
if not relu_names: concerns.append("No encoder ReLU modules found; inspect functional activations manually.")
print("Conversion concerns:", concerns or "No listed module-level blockers found.")
print("Always verify normalization, decoder operations and both tuple-action logits after extraction.")
inspection = {"checkpoint":str(CKPT_PATH), "architecture":str(ac),
              "trainable_parameters":sum(p.numel() for p in ac.parameters() if p.requires_grad),
              "layer_shapes":shapes, "encoder_relus":relu_names, "batchnorm":batchnorm_names,
              "other_activations":other_activations, "maxpool":maxpool_names,
              "concerns":concerns, "action_factors":[3,2]}
inspection_dir = OUTPUT_ROOT / "results" / EXPERIMENT
inspection_dir.mkdir(parents=True,exist_ok=True)
(inspection_dir / "ann_architecture.json").write_text(json.dumps(inspection,indent=2))


## Fixed-seed gameplay evaluation

Return and decisions survived are recorded. Decisions are not raw game tics (frame skip is 4). No doom_basic thresholds or kill counts inferred from rewards. These seeds are for validation; use a separate range for final held-out reporting.

In [ ]:
# Deterministic actions on the identical episode-seed list in both notebooks.
env = create_env(ENV_NAME, cfg=eval_cfg, env_config=None)
records = []
try:
    for ep_seed in EVAL_SEEDS:
        obs, _ = env.reset(seed=ep_seed)
        done = truncated = False; total = 0.; decisions = 0
        while not (done or truncated):
            action = tuple_action(policy_logits(obs))
            obs, reward, done, truncated, info = env.step(action)
            total += float(reward); decisions += 1
        records.append({"seed": ep_seed, "return": total, "decisions": decisions,
                        "terminated": bool(done), "truncated": bool(truncated)})
        if len(records) % 10 == 0: print("Episodes:", len(records))
finally:
    env.close()
rewards = np.array([r["return"] for r in records])
print("Mean return:", rewards.mean(), "std:", rewards.std(), "SE:", rewards.std()/np.sqrt(len(rewards)))
print("Mean decisions survived:", np.mean([r["decisions"] for r in records]))
results_dir = OUTPUT_ROOT / "results" / EXPERIMENT
results_dir.mkdir(parents=True, exist_ok=True)
report = {"experiment": EXPERIMENT, "env": ENV_NAME, "checkpoint": str(CKPT_PATH),
          "checkpoint_sha256": hashlib.file_digest(CKPT_PATH.open("rb"), "sha256").hexdigest(),
          "env_steps": int(ckpt["env_steps"]), "mean_return": float(rewards.mean()),
          "std_return": float(rewards.std()), "episodes": records,
          "packages": {p: version(p) for p in ["torch","numpy","sample-factory","vizdoom","gymnasium"]},
          "python": sys.version, "gpu": torch.cuda.get_device_name(0),
          "resolved_training_config": json.loads((RUN_DIR / "config.json").read_text())}
(results_dir / "evaluation.json").write_text(json.dumps(report, indent=2))
np.save(results_dir / "rewards.npy", rewards)
print("Saved:", results_dir)


## Interpretation and limitations

Compare ANN and LIF at equal training budgets using the same evaluation seeds, and repeat training across several seeds.
The environment's return scale differs from doom_basic. A 100k smoke test checks execution, not convergence.
This notebook does not convert the ANN, estimate energy, or extract a standalone policy. The full checkpoint
preserves weights, observation statistics and the critic for future use. Preserve config.json alongside it.
GPU/package compatibility and complete gameplay training still need validation on the university server.
Seeding does not guarantee identical asynchronous training across processes or software versions.


## Replay the best training checkpoint
Runs three deterministic gameplay episodes and embeds MP4 videos. Uses `best_*.pth`,
separately from the latest-checkpoint evaluation above. A short smoke test might not
produce a best checkpoint yet. ffmpeg with libx264 support is required. Frames stream
to the encoder instead of accumulating a long video in RAM. Paths with spaces are supported.
Replays show the best recorded training policy; they do not establish which checkpoint
generalizes best. Rerunning the cell overwrites these seed-specific videos.


In [ ]:
# Replay the checkpoint with the highest recorded training reward.
# This is Sample Factory's best snapshot, not a held-out ranking of every model.
import re, shutil, subprocess, copy
from IPython.display import Video, display

REPLAY_EPISODES = 3
REPLAY_FIRST_SEED = 20000
ffmpeg = shutil.which("ffmpeg")
if ffmpeg is None:
    raise RuntimeError("Replay needs ffmpeg. Load your cluster's ffmpeg module, then rerun this cell.")
best_paths = list((RUN_DIR / "checkpoint_p0").glob("best_*.pth"))
if not best_paths:
    raise FileNotFoundError("No best-reward checkpoint saved yet. Continue training; replay does not silently use latest.")
def recorded_reward(path):
    match = re.search(r"_reward_([-+0-9.eE]+)\.pth$", path.name)
    if match is None:
        raise ValueError(f"Cannot read best reward from {path.name}")
    return float(match.group(1)), path.name
BEST_PATH = max(best_paths, key=recorded_reward)
best_ckpt = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
replay_model = create_actor_critic(eval_cfg, OBS_SPACE, ACTION_SPACE)
replay_model.load_state_dict(best_ckpt["model"], strict=True)
replay_model.model_to_device(torch.device(DEVICE)); replay_model.eval()
replay_cfg = copy.deepcopy(eval_cfg)
replay_env = create_env(ENV_NAME, cfg=replay_cfg, env_config=None, render_mode="rgb_array")
replay_dir = OUTPUT_ROOT / "replays" / EXPERIMENT
replay_dir.mkdir(parents=True, exist_ok=True)
replay_records = []
try:
    for episode in range(REPLAY_EPISODES):
        seed = REPLAY_FIRST_SEED + episode
        video_path = replay_dir / f"best_seed{seed}.mp4"
        log_path = replay_dir / f"best_seed{seed}_ffmpeg.log"
        obs, _ = replay_env.reset(seed=seed)
        done = truncated = False; reward_sum = 0.; frames_written = 0
        encoder = None; log_file = log_path.open("wb")
        try:
            while not (done or truncated):
                frame = replay_env.render()
                if frame is None: raise RuntimeError("rgb_array rendering returned no frame")
                frame = np.asarray(frame)
                if frame.ndim == 3 and frame.shape[0] == 3 and frame.shape[-1] != 3:
                    frame = frame.transpose(1, 2, 0)
                if frame.ndim != 3 or frame.shape[-1] != 3:
                    raise ValueError(f"Unexpected replay frame shape {frame.shape}")
                frame = np.ascontiguousarray(frame, dtype=np.uint8)
                if encoder is None:
                    height, width = frame.shape[:2]
                    # One captured image per environment decision; Doom runs at 35 tics/s.
                    fps = 35.0 / replay_cfg.env_frameskip
                    encoder = subprocess.Popen([
                        ffmpeg, "-y", "-f", "rawvideo", "-pix_fmt", "rgb24",
                        "-s", f"{width}x{height}", "-r", str(fps), "-i", "pipe:0",
                        "-an", "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
                        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
                        str(video_path)], stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=log_file)
                encoder.stdin.write(frame.tobytes()); frames_written += 1
                raw_dict = obs if isinstance(obs, dict) else {"obs": obs}
                batch = {k: torch.as_tensor(v).unsqueeze(0) for k,v in raw_dict.items()}
                with torch.no_grad():
                    normalized = prepare_and_normalize_obs(replay_model, batch)
                    core, _ = replay_model.forward_core(replay_model.forward_head(normalized), None)
                    logits = replay_model.forward_tail(core, values_only=False, sample_actions=False)["action_logits"]
                obs, reward, done, truncated, _ = replay_env.step(tuple_action(logits))
                reward_sum += float(reward)
        finally:
            try:
                if encoder is not None:
                    try:
                        encoder.stdin.close()
                    finally:
                        try: encoder.wait(timeout=60)
                        except subprocess.TimeoutExpired:
                            encoder.kill(); encoder.wait()
                            raise RuntimeError(f"ffmpeg stalled; see {log_path}")
            finally:
                log_file.close()
        if encoder is None or encoder.returncode != 0 or not video_path.is_file() or video_path.stat().st_size == 0:
            raise RuntimeError(f"Video encoding failed; inspect {log_path}")
        replay_records.append({"seed": seed, "return": reward_sum, "frames": frames_written,
                               "video": str(video_path)})
        print("Replay:", video_path, "return:", reward_sum)
        display(Video(str(video_path), embed=True, width=640))
finally:
    replay_env.close()
    del replay_model
    torch.cuda.empty_cache()
with BEST_PATH.open("rb") as checkpoint_file:
    checkpoint_hash = hashlib.file_digest(checkpoint_file, "sha256").hexdigest()
(replay_dir / "replay_metadata.json").write_text(json.dumps({
    "checkpoint": str(BEST_PATH), "checkpoint_sha256": checkpoint_hash,
    "env_steps": int(best_ckpt["env_steps"]), "selection": "highest recorded training reward",
    "frame_skip": replay_cfg.env_frameskip, "episodes": replay_records}, indent=2))
